In [20]:
import os
import sys

import numpy as np

sys.path.append(os.path.abspath(os.path.join("..", "src")))
import tomllib

import gramian_obs
import utils
import patient_cohort

In [21]:
with open("../config.toml", "rb") as f:
    cfg = tomllib.load(f)

root_module = "global"
sim_hours = cfg[root_module]["sim_hours"]
dt = cfg[root_module]["dt"]
dtmeas = cfg[root_module]["dtmeas"]
meas_noise_std = cfg[root_module]["meas_noise_std"]
uex_bounds = cfg[root_module]["uex_bounds"]
D_bounds = cfg[root_module]["D_bounds"]
PN_bounds = cfg[root_module]["PN_bounds"]
SI_params = cfg[root_module]["SI_params"]
y0 = cfg[root_module]["y0"]
x_max = cfg[root_module]["x_max"]
num_patients = 1

curr_module = "01"
n_pert = cfg[curr_module]["n_pert"]
c = cfg[curr_module]["c"]
threshold = cfg[curr_module]["threshold"]

PatientCohort = patient_cohort.PatientCohort(
    num_patients=num_patients,
    sim_hours=sim_hours,
    dt=dt,
    dtmeas=dtmeas,
    meas_noise_std=meas_noise_std,
    uex_bounds=uex_bounds,
    D_bounds=D_bounds,
    PN_bounds=PN_bounds,
    SI_params=SI_params,
    y0=y0,
    SI_piecewise_changes=None,
)

patient_data = PatientCohort.patient_data()

x0 = patient_data[0]["x0"]
uen = patient_data[0]["uen"]
uex_const = patient_data[0]["uex"][0]
D_const = patient_data[0]["D"][0]
PN_const = patient_data[0]["PN"][0]
SI_const = patient_data[0]["SI"][0]

In [22]:
t = np.arange(0, sim_hours * 60 + 1, dt)  # 0 to 60 hours with minute-level resolution
uex_func = utils.gen_uex_func(uex_const=uex_const)
PN_func = utils.gen_PN_func(PN_const=PN_const)
D_func = utils.gen_D_func(D_const=D_const)
SI_func = utils.gen_SI_func(SI_const=SI_const)

Gramian = gramian_obs.EmperialGramianMatrix(
    x0=x0,
    uen=uen,
    n_pert=n_pert,
    c=c,
    x_max=x_max,
    t=t,
    dt=dt,
    uex_func=uex_func,
    PN_func=PN_func,
    D_func=D_func,
    SI_func=SI_func,
    threshold=threshold,
    SI_est=None,
)
W_O, eig_vals, eig_vectors = Gramian.gramian()

In [23]:
eig_vals # / np.linalg.norm(eig_vals)

array([1.47770372e-08, 1.91269439e-04, 2.01027635e-03, 2.42892674e-01,
       4.41611156e+01])